In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Values and the Bellman equation

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part A (value-based methods), session 1**
    - the tour gave the whole picture and named three things an agent can learn: a model, a value, a policy
    - this session takes the **value**: it turns "best behavior" into numbers we can write equations for
    - the next session solves those equations exactly when the environment's rules are known
- **The question:** how could we ever compute "the best behavior"?
- **By the end you can**
    - say what a **value** is (one number for a state) and what the **value function** $\statevalfun_\pi$ is (the rule giving it for every state), and read a value table
    - write the Bellman equation, in its averaging form and its optimality (max) form, and say what a backup is
    - check the optimality equation by hand at a square of Frozen Lake
    - say why values give a policy (a one-step lookahead that uses the model)
    - say why the Bellman equation is not yet a method: the value is on both sides


# What is a value?

**Why this section?** The tour named three things an agent can learn. We start with the *value*: one number per state that says how good it is to be there.

Recall the return $\rlret_\tt = \sum_{k \geq 0} \disc^k \, \rewseq_{\tt+k+1}$ (the discounted rewards from time $\tt$ on; the *return-to-go*). Value-based methods revolve around the idea of
- creating a **state value function** $\statevalfun_\pi(\state)$ 
- to assign a value to each state $\state$ in the set of states $\States$
$$
\statevalfun_{\pi}: \States \to \Reals
$$
(one real number, in $\Reals$, per state)

$\statevalfun_\pi(\state)$ is the expectation 
$$
\Exp{\pi}\left[ \rlret_\tt \mid \stateseq_\tt = \state \right]
$$
the expected return from state $\state$ when following $\pi$

- if nothing is random (no chance in the environment or in $\pi$), the expectation is just the return of the one path that results

**Value and value function**

- the **value** of a state is that one number, $\statevalfun_\pi(\state)$
- the **value function** is the whole assignment $\statevalfun_\pi$, state by state
- below, "value" means the number and "value function" means the rule that gives it
- when the states are few, the value function is stored as a **table**: one entry per state (next slides)


**Frozen Lake in this session**

- the grid has $16$ squares, numbered $0$ to $15$ row by row from the top left (**S** is square $0$, **G** is square $15$, **F** are frozen squares, **H** holes)
- four actions: left, down, right, up; **non-slippery** here: every move goes where intended (a simplification of the real, slippery lake of the tour, so the numbers differ from the tour's slippery model panel)
- reward: $+1$ when a move enters **G**, $0$ otherwise; the episode ends at **G** or at a hole **H**
- discount $\disc = 0.9$
- terminal squares (**G** and the holes) have value $0$: the $+1$ is earned by the move into **G**, not by being there


**The model of this lake**

- the **model** is the environment's rule for what happens after an action: $\transp(\state', \rew \mid \state, \act)$, the chance that taking action $\act$ in square $\state$ leads to square $\state'$ with reward $\rew$ (the book's joint form, used throughout the series)
- on this non-slippery lake one pair $(\state', \rew)$ has probability $1$ and every other pair $0$
- a move into the outer wall leaves the agent where it is


We usually represent (and implement) $\statevalfun_\pi(\state)$ as a table.
- for the case of finite number of states

For Frozen Lake a table with every value set to $0$ (the starting point of the methods that follow) looks like:

<br>
<table>
    <center><strong>Value function (uninitialized) as a table</strong></center>
<tr>
    <td><img src="images/Frozen_Lake_env.png"></td>
    <td><img src="images/value_table_zero.png"></td>
</tr>
</table>

Each cell
- corresponds to a state
- contains a real-valued expected value of return-to-go from that state
    - in the pictures, color shows magnitude: pale is $0$, deeper color is higher
    
In Frozen Lake
- there is a single reward of +1
    - hence: expected returns in each cell are in the range $[0,1]$
- earned by the move into the terminal state (G)

# Interlude: the Bellman recursion and backups

**Why this section?** The value of a state is tied to the values of the states that follow it. That relation, the Bellman equation, drives every method in this series; we meet it once, carefully, here.

- the return satisfies a recursion: $\rlret_\tt = \rewseq_{\tt+1} + \disc\,\rlret_{\tt+1}$
    - the return from now is the next reward plus the discounted return from wherever we land
    - this is the idea of a **backup**: a quantity now is set from the same quantity one step later
- so the *expected* return from a state is the expected first reward plus $\disc$ times the expected value of the next state: the **Bellman equation**
- a **backup** is an update that sets a state's value from the values of the states that can follow it: information flows from successor to predecessor
    - the return recursion is a backup along *one sampled path*; the Bellman equation is the backup *averaged* over everything that can happen (sampling versus averaging returns in later sessions)
    - repeated backups carry the reward back along the route, one step at a time, from states near the reward to states far from it
- this idea is common to every value-based method of the series
    - dynamic programming applies it exactly with a known model (next session)
    - sampling methods apply it to sampled experience (the session after)


The Bellman equation says that the value of a state is an average over what can happen next
$$
\statevalfun_\pi(\state) = \sum_{\act} \pi(\act \mid \state) \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_\pi(\state')\bigr)
$$

- **why two sums?** two things can be random
    - the policy may be random: $\pi(\act \mid \state)$ weights the actions
    - the environment may be random: the same action in the same state can lead to different next states $\state'$ (and rewards $\rew$), weighted by $\transp(\state', \rew \mid \state, \act)$
- example: on the real, *slippery* lake a move goes the intended way one time in three and slides to a side otherwise, so up to three successor squares each have weight $1/3$: the inner sum is a true average
- on today's non-slippery lake every move has one successor (weight $1$), so the inner sum has a single term


**Reading the terms of the equation**

- the immediate reward $\rew$ comes with the move: it is part of the outcome that the model gives a probability, together with the next state $\state'$
- plus the discounted (by $\disc$) value $\statevalfun_\pi(\state')$ of the successor state
- why this is right: the return satisfies $\rlret_\tt = \rewseq_{\tt+1} + \disc\,\rlret_{\tt+1}$, so the expected return from $\state$ is the expected first reward plus $\disc$ times the expected return from wherever we land


**Two forms of the equation: average over actions, or choose the best**

- **for a given policy $\pi$** (the equation above): the policy decides, so the value *averages* over the actions it might take
- **for the best behavior** (the *optimality* form): the best policy picks the best action in every state, so the average is replaced by a **max**

$$
\statevalfun_*(\state) = \max{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_*(\state')\bigr)
$$

- $\statevalfun_\pi$: how good is this state *if we follow $\pi$*; $\statevalfun_*$: how good is it *under the best behavior*
- both are one equation per state, with the value of the successor states on the right-hand side
- the optimality form is what "computing the best behavior" will mean: find the $\statevalfun_*$ that satisfies it


**Where this equation comes back**

- the first (averaging) form is the template for the whole of Part A; each later session modifies it
    - **A2, dynamic programming:** the equation becomes an *update rule*, applied to every state again and again
    - **A3, sampling:** the average over $\transp$ is replaced by *one sampled* reward and next state
    - **A4, control:** the same equation written for action values $\actvalfun$, with the max form for the best behavior
    - **A5, deep Q-networks:** the right-hand side becomes the *target* a network is trained to match
- so it is worth being sure of this one: the later equations are this one with a part replaced


## From values to a policy

**Why this section?** The point of values is acting. A value table gives a policy by looking one step ahead.

- the value of a state says how good it is; to *act*, we need to choose an action


Given the optimal value function $\statevalfun_*$, an optimal deterministic policy looks one step ahead

$$
\pi_*(\state) = \argmax{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_*(\state')\bigr)
$$

- From state $\state$
- try each action $\act$
- use the model $\transp$ to weigh each possible next state $\state'$
- choose the action with the best reward plus discounted value of what follows

Note: the $\argmax{}$ results in a *deterministic* policy.

Without the model this lookahead is impossible; action values (session A4, on control) make the last step model-free.


Here is the answer we are after: the optimal values and policy (non-slippery, discount $0.9$), as computed by the method of the next session (the picture's title mentions its sweeps)
- $\statevalfun_*(\state)$ 
- and an optimal policy $\pi_*(\state)$, shown as arrows

<table>
    <center><strong>Value function (converged) as a table</strong></center>
<tr>
    <td><img src="images/value_iteration_final.png"></td>
</tr>
</table>

Where two actions tie (squares $0$ and $9$), one arrow is shown.


## What "a model" means here

**Why this section?** The lookahead above and the Bellman equation use the environment's rules. What changes if we do not know them?

- the **model** is the environment's rule $\transp(\state', \rew \mid \state, \act)$: what happens after each action (the tour's "a model" to learn is this same object)
- here *model-based* means the model is **known** and used for planning: nothing needs to be tried out, all the information is in the model (next session)
- *model-free* means we only get **samples** by interacting with the environment (the session after)
- this is narrower than the common use of "model-based", which also includes *learning* the model from experience; this series does not cover learning a model


## Checking the optimality equation by hand

**Why this section?** The equation should hold for the optimal values. Check it at two squares, using the optimal values shown earlier in this session (non-slippery, $\disc = 0.9$).

- square $10$: its neighbors are $9$ (left), $6$ (up), $11$ (right, a hole) and $14$ (down); $\statevalfun_*(9) = \statevalfun_*(6) = 0.81$, $\statevalfun_*(14) = 1$, a hole has value $0$
    - down: $0 + 0.9 \times 1 = 0.9$
    - left: $0 + 0.9 \times 0.81 = 0.729$; up: also $0.729$
    - right (into the hole): $0 + 0.9 \times 0 = 0$
    - the max is $0.9$, which equals $\statevalfun_*(10)$: the equation holds


**The check at square 6, and at the start**

- square $6$: neighbors $2$ (up, $0.729$), $10$ (down, $0.9$), and holes $5$, $7$
    - down: $0.9 \times 0.9 = 0.81$; up: $0.9 \times 0.729 \approx 0.656$; into a hole: $0$
    - the max is $0.81 = \statevalfun_*(6)$
- each such backup moves value one square back from the goal, shrunk by $\disc$
- under the best behavior the start is a $6$-move route: the $+1$ arrives on the 6th move, discounted five times, so $\statevalfun_*(\text{start}) = 0.9^5 \approx 0.59$


## The limitation: the equation is not yet an answer

**Why this section?** We can write down what the best behavior must satisfy, and check it when given the answer, but we have not computed it.

- the Bellman equation has the value on **both sides**: $\statevalfun(\state)$ depends on the values of the successors, which depend on theirs
- so we cannot just evaluate it
- the idea: **iterate**
    - start from a table of zeros (the table of the first section)
    - apply the backup to every square, again and again
    - each pass carries the reward one square further back; the sequence of tables $\statevalfun_0, \statevalfun_1, \ldots$ should approach $\statevalfun_*$
- how well this works, and for which equation (average or max), is the next session


# What we established, and what it sets up

- **Established**
    - the state value $\statevalfun_\pi(\state)$ is the expected return from a state; on a small problem it is a table
    - the **Bellman equation** relates a state's value to its successors' values; a **backup** carries value from successors to predecessors
    - two forms: an average over the policy's actions for $\statevalfun_\pi$, and a max for the best behavior $\statevalfun_*$; we checked the max form by hand
    - a value table gives a policy by a one-step lookahead that uses the model
- **Sets up (the limitation)**
    - the equation has the value on both sides, so it is not yet a method: next session solves it by iterating (dynamic programming), when the model is known
    - then: what if the model is not known?
